In [3]:
# 📊 Nexus GraphRAG — Evaluation Notebook

Compares **Naive RAG** (vector-only Qdrant search + Groq synthesis) vs **Hybrid GraphRAG** (our full ReAct agent API) using [Ragas](https://docs.ragas.io) metrics on 5 complex financial questions from Apple & Microsoft 10-K filings.

| Component | Technology |
|---|---|
| Eval LLM | Groq `llama-3.3-70b-versatile` |
| Naive RAG | Direct Qdrant vector search + Groq answer synthesis |
| Hybrid RAG | `POST http://localhost:8000/ask` (ReAct agent) |
| Metrics | `answer_relevancy`, `answer_correctness` |

> ⚠️ **Before running:** make sure `docker compose up -d` is running so the API is live at `http://localhost:8000`.


SyntaxError: invalid decimal literal (3334107515.py, line 7)

In [23]:
# ── Cell 2: Imports & configuration ──────────────────────────────────────────
import sys, os, time, json, re, textwrap, warnings, importlib
import requests
import pandas as pd
from pathlib import Path

warnings.filterwarnings("ignore", category=DeprecationWarning)

# Make src/ importable from inside the notebook (no __file__ in Jupyter)
PROJECT_ROOT = Path(os.getcwd()).parent   # evals/ → project root
sys.path.insert(0, str(PROJECT_ROOT))

from src import config
# Force reload to pick up new ANTHROPIC_API_KEY from config.py
importlib.reload(config)

# Ragas 0.4.x — use collections import path to avoid deprecation warnings
from langchain_anthropic import ChatAnthropic
from ragas.llms import LangchainLLMWrapper
from ragas import evaluate
try:
    from ragas.metrics.collections import answer_relevancy, answer_correctness
except ImportError:
    from ragas.metrics import answer_relevancy, answer_correctness
from datasets import Dataset

# Direct SDKs — no llama-index, Python 3.14 safe
from groq import Groq as GroqClient
from qdrant_client import QdrantClient
from fastembed import TextEmbedding

# ── Validate keys ─────────────────────────────────────────────────────────────
assert config.GROQ_API_KEY, "❌ GROQ_API_KEY not set in .env"
assert config.ANTHROPIC_API_KEY, "❌ ANTHROPIC_API_KEY not set in .env"

GROQ_MODEL      = "llama-3.3-70b-versatile"
ANTHROPIC_MODEL = "claude-3-5-haiku-20241022"
EMBED_MODEL     = "BAAI/bge-small-en-v1.5"
COLLECTION      = "sec_10k_filings"
API_URL         = "http://localhost:8000/ask"
GROQ_SLEEP      = 2.0   # seconds between LLM calls (free-tier rate limit)

print("✅ Imports OK")
print(f"   Project root       : {PROJECT_ROOT}")
print(f"   Groq model (Naive) : {GROQ_MODEL}")
print(f"   Anthropic model    : {ANTHROPIC_MODEL}")
print(f"   API endpoint       : {API_URL}")


✅ Imports OK
   Project root       : /Users/ramunalla/Personal/Projects/nexus-graphRAG
   Groq model (Naive) : llama-3.3-70b-versatile
   Anthropic model    : claude-3-5-haiku-20241022
   API endpoint       : http://localhost:8000/ask


In [12]:

# ── Cell 3: Connect to services ───────────────────────────────────────────────
print("Connecting to Groq, Qdrant and loading embedding model…")

groq_client  = GroqClient(api_key=config.GROQ_API_KEY)
qdrant_client = QdrantClient(url=config.QDRANT_URL)
embedder     = TextEmbedding(model_name=EMBED_MODEL)

# Sanity-check the API is reachable
try:
    r = requests.get("http://localhost:8000/health", timeout=5)
    health = r.json()
    print(f"✅ API health: {health['status']} | agent: {health['agent']}")
except Exception as e:
    print(f"⚠️  API not reachable ({e}). Make sure 'docker compose up -d' is running.")

print("✅ Qdrant connected | embedding model loaded")


Connecting to Groq, Qdrant and loading embedding model…
✅ API health: operational | agent: ready
✅ Qdrant connected | embedding model loaded


In [6]:

# ── Cell 4: Q&A dataset ───────────────────────────────────────────────────────
qa_pairs = [
    (
        "What major acquisitions did Microsoft make recently, and how did they impact the gaming division?",
        "Microsoft acquired Activision Blizzard, which significantly expanded its gaming division, "
        "adding major franchises and boosting Xbox content and services revenue."
    ),
    (
        "How does Apple rely on third-party manufacturing, and what risks are associated with it?",
        "Apple relies heavily on outsourcing manufacturing to partners primarily in Asia. "
        "Risks include supply chain disruptions, geopolitical tensions, and reliance on single-source suppliers."
    ),
    (
        "What are Microsoft's primary revenue streams within its Intelligent Cloud segment?",
        "The Intelligent Cloud segment revenue is primarily driven by Azure and other cloud services, "
        "server products, and enterprise services."
    ),
    (
        "Who are Apple's main competitors in the smartphone and wearables market?",
        "Apple competes against other global tech companies like Samsung and Google in smartphones, "
        "and various fitness and smart device makers in wearables."
    ),
    (
        "Describe the regulatory risks Microsoft faces regarding data privacy and AI.",
        "Microsoft faces complex regulatory risks including GDPR in Europe, antitrust scrutiny over "
        "its AI partnerships, and emerging AI safety regulations globally."
    ),
]

questions    = [q for q, _ in qa_pairs]
ground_truths = [a for _, a in qa_pairs]

print(f"✅ Dataset ready — {len(questions)} questions loaded")
for i, q in enumerate(questions, 1):
    print(f"  Q{i}: {q[:80]}…")


✅ Dataset ready — 5 questions loaded
  Q1: What major acquisitions did Microsoft make recently, and how did they impact the…
  Q2: How does Apple rely on third-party manufacturing, and what risks are associated …
  Q3: What are Microsoft's primary revenue streams within its Intelligent Cloud segmen…
  Q4: Who are Apple's main competitors in the smartphone and wearables market?…
  Q5: Describe the regulatory risks Microsoft faces regarding data privacy and AI.…


In [7]:

# ── Cell 5: Naive RAG helper (direct Qdrant + Groq, no llama-index) ───────────
def naive_rag_answer(question: str) -> str:
    """
    Naive RAG baseline:
      1. Embed the question with fastembed
      2. Retrieve top-5 chunks from Qdrant
      3. Ask Groq to answer using only those chunks
    """
    # Step 1 — embed
    q_vector = list(embedder.embed([question]))[0].tolist()

    # Step 2 — retrieve (qdrant-client 1.7+ uses query_points)
    try:
        response = qdrant_client.query_points(
            collection_name=COLLECTION,
            query=q_vector,
            limit=5,
        )
        hits = response.points
    except AttributeError:
        hits = qdrant_client.search(
            collection_name=COLLECTION,
            query_vector=q_vector,
            limit=5,
        )

    if not hits:
        return "No relevant context found in the vector store."

    context = "\n\n---\n\n".join(
        f"[{h.payload.get('source','?')}] {h.payload.get('text','')}" for h in hits
    )

    # Step 3 — generate answer
    prompt = textwrap.dedent(f"""\
        Answer the question using ONLY the provided context. Be concise and factual.

        Context:
        {context}

        Question: {question}
        Answer:""")

    resp = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=512,
    )
    return resp.choices[0].message.content.strip()


print("✅ naive_rag_answer() defined")


✅ naive_rag_answer() defined


In [8]:

# ── Cell 6: Run both systems against all questions ────────────────────────────
naive_answers  = []
hybrid_answers = []

for i, question in enumerate(questions, 1):
    print(f"\n{'─'*60}")
    print(f"Q{i}: {question}")

    # ── A: Naive RAG ──────────────────────────────────────────
    print("  [Naive RAG] querying…", end=" ", flush=True)
    try:
        naive_ans = naive_rag_answer(question)
        naive_answers.append(naive_ans)
        print(f"done ✅  ({len(naive_ans)} chars)")
    except Exception as e:
        naive_answers.append(f"Error: {e}")
        print(f"❌ {e}")

    time.sleep(GROQ_SLEEP)

    # ── B: Hybrid GraphRAG API ─────────────────────────────────
    print("  [Hybrid GraphRAG] querying…", end=" ", flush=True)
    try:
        api_resp = requests.post(
            API_URL,
            json={"question": question},
            timeout=120,
        )
        if api_resp.status_code == 200:
            hybrid_ans = api_resp.json().get("answer", "No answer field")
            hybrid_answers.append(hybrid_ans)
            print(f"done ✅  ({len(hybrid_ans)} chars)")
        else:
            hybrid_answers.append(f"API HTTP {api_resp.status_code}")
            print(f"❌ HTTP {api_resp.status_code}")
    except Exception as e:
        hybrid_answers.append(f"API Connection Error: {e}")
        print(f"❌ {e}")

    time.sleep(GROQ_SLEEP)

print(f"\n\n✅ All {len(questions)} questions answered by both systems.")



────────────────────────────────────────────────────────────
Q1: What major acquisitions did Microsoft make recently, and how did they impact the gaming division?
  [Naive RAG] querying… done ✅  (131 chars)
done ✅  (131 chars)
  [Hybrid GraphRAG] querying…   [Hybrid GraphRAG] querying… done ✅  (705 chars)

────────────────────────────────────────────────────────────
Q2: How does Apple rely on third-party manufacturing, and what risks are associated with it?
  [Naive RAG] querying… done ✅  (266 chars)
  [Hybrid GraphRAG] querying… done ✅  (55 chars)

────────────────────────────────────────────────────────────
Q3: What are Microsoft's primary revenue streams within its Intelligent Cloud segment?
  [Naive RAG] querying… done ✅  (534 chars)
  [Hybrid GraphRAG] querying… done ✅  (317 chars)

────────────────────────────────────────────────────────────
Q4: Who are Apple's main competitors in the smartphone and wearables market?
  [Naive RAG] querying… done ✅  (293 chars)
  [Hybrid GraphRAG

In [24]:
# ── Cell 7: Ragas evaluation ──────────────────────────────────────────────────
import subprocess, sys, json, tempfile, os, time

# Guard
_required = ["questions", "naive_answers", "hybrid_answers", "ground_truths", "config", "ANTHROPIC_MODEL"]
_missing  = [v for v in _required if v not in globals()]
if _missing:
    raise RuntimeError(f"❌ Variables not in scope: {_missing}\n   Run cells 2 → 6 first.")

# ── Why only answer_correctness? ──────────────────────────────────────────────
# answer_relevancy internally calls LLM with n>1 (multi-completion), which
# Groq does not support → BadRequestError → silent 0.
# answer_correctness uses standard single completions → fully compatible.
# We compute a semantic similarity proxy manually using our fastembed model.

# Replace short / error answers with neutral stub (scores 0, not nan)
FALLBACK_STUB = "No answer available."
def _clean(answers):
    return [a if (a and len(a) > 60
                  and not a.startswith("Error")
                  and not a.startswith("API")
                  and not a.startswith("No relevant"))
            else FALLBACK_STUB
            for a in answers]

clean_naive  = _clean(naive_answers)
clean_hybrid = _clean(hybrid_answers)

replaced_naive  = sum(1 for a, b in zip(naive_answers,  clean_naive)  if a != b)
replaced_hybrid = sum(1 for a, b in zip(hybrid_answers, clean_hybrid) if a != b)
if replaced_naive or replaced_hybrid:
    print(f"⚠️  Stubbed {replaced_naive} Naive / {replaced_hybrid} Hybrid low-quality answers (will score 0)")


def run_ragas_subprocess(qs, answers, gts, anthropic_api_key, anthropic_model, label=""):
    """
    Subprocess-isolated ragas run.
    Uses only answer_correctness (LLM judge, n=1, Anthropic-compatible).
    Semantic similarity is computed via cosine of fastembed vectors instead
    of answer_relevancy (which requires n>1 and breaks on Groq).
    """
    script = (
        "import warnings, json\n"
        "warnings.filterwarnings('ignore')\n"
        "import numpy as np\n"
        "from langchain_anthropic import ChatAnthropic\n"
        "from ragas.llms import LangchainLLMWrapper\n"
        "from ragas.embeddings import BaseRagasEmbeddings\n"
        "from ragas.run_config import RunConfig\n"
        "from ragas.metrics import answer_correctness\n"
        "from ragas import evaluate\n"
        "from datasets import Dataset\n"
        "from fastembed import TextEmbedding\n"
        "\n"
        "class FastEmbedRagas(BaseRagasEmbeddings):\n"
        "    run_config = RunConfig()\n"
        "    def __init__(self):\n"
        "        super().__init__()\n"
        "        self._m = TextEmbedding('BAAI/bge-small-en-v1.5')\n"
        "    def embed_query(self, t): return list(self._m.embed([t]))[0].tolist()\n"
        "    def embed_documents(self, ts): return [v.tolist() for v in self._m.embed(ts)]\n"
        "    async def aembed_query(self, t): return self.embed_query(t)\n"
        "    async def aembed_documents(self, ts): return self.embed_documents(ts)\n"
        "\n"
        f"questions     = {json.dumps(qs)}\n"
        f"answers       = {json.dumps(answers)}\n"
        f"ground_truths = {json.dumps(gts)}\n"
        "\n"
        f"llm = LangchainLLMWrapper(ChatAnthropic(model='{anthropic_model}', api_key='{anthropic_api_key}', temperature=0))\n"
        "emb = FastEmbedRagas()\n"
        "\n"
        "# --- answer_correctness (LLM judge, n=1) ---\n"
        "ds = Dataset.from_dict({'question': questions, 'answer': answers, 'ground_truth': ground_truths})\n"
        "r  = evaluate(ds, metrics=[answer_correctness], llm=llm, embeddings=emb)\n"
        "df = r.to_pandas()\n"
        "ac = round(float(df['answer_correctness'].fillna(0).mean()), 4)\n"
        "\n"
        "# --- semantic_similarity: cosine(answer_vec, ground_truth_vec) ---\n"
        "m = FastEmbedRagas()._m\n"
        "ans_vecs = list(m.embed(answers))\n"
        "gt_vecs  = list(m.embed(ground_truths))\n"
        "sims = []\n"
        "for av, gv in zip(ans_vecs, gt_vecs):\n"
        "    a, b = np.array(av), np.array(gv)\n"
        "    sims.append(float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9)))\n"
        "ss = round(float(np.mean(sims)), 4)\n"
        "\n"
        "print(json.dumps({'semantic_similarity': ss, 'answer_correctness': ac}))\n"
    )

    with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False, encoding="utf-8") as f:
        f.write(script)
        tmp = f.name

    print(f"   ⏳ [{label}] running…", flush=True)
    try:
        result = subprocess.run(
            [sys.executable, tmp],
            capture_output=True, text=True, timeout=360,
            env={**os.environ, "PYTHONWARNINGS": "ignore"},
        )
        if result.returncode != 0:
            real_err = "\n".join(
                l for l in result.stderr.splitlines()
                if l.strip() and not any(w in l for w in
                   ("UserWarning", "FutureWarning", "DeprecationWarning",
                    "warnings.warn", "google.generativeai"))
            )
            raise RuntimeError(f"Subprocess failed (exit {result.returncode}):\n{real_err[-2000:]}")
        lines = [l for l in result.stdout.strip().splitlines() if l.strip()]
        return json.loads(lines[-1])
    finally:
        os.unlink(tmp)


print("🚀 Ragas evaluation — subprocess mode (using Anthropic Claude Haiku)\n")
print("   Metrics:")
print("   • answer_correctness   — LLM judge (Claude Haiku 3.5, n=1) ✅")
print("   • semantic_similarity  — cosine(answer, ground_truth) via fastembed ✅\n")

print("1/2  Naive RAG …")
naive_scores = run_ragas_subprocess(
    questions, clean_naive, ground_truths,
    config.ANTHROPIC_API_KEY, ANTHROPIC_MODEL, label="Naive RAG"
)
print(f"     ✅ {naive_scores}\n")

time.sleep(5)   # brief pause between batch calls

print("2/2  Hybrid GraphRAG …")
hybrid_scores = run_ragas_subprocess(
    questions, clean_hybrid, ground_truths,
    config.ANTHROPIC_API_KEY, ANTHROPIC_MODEL, label="Hybrid GraphRAG"
)
print(f"     ✅ {hybrid_scores}\n")

print("✅ Done — run the next cell for the comparison table.")


⚠️  Stubbed 0 Naive / 2 Hybrid low-quality answers (will score 0)
🚀 Ragas evaluation — subprocess mode (using Anthropic Claude Haiku)

   Metrics:
   • answer_correctness   — LLM judge (Claude Haiku 3.5, n=1) ✅
   • semantic_similarity  — cosine(answer, ground_truth) via fastembed ✅

1/2  Naive RAG …
   ⏳ [Naive RAG] running…
     ✅ {'semantic_similarity': 0.6653, 'answer_correctness': 0.0}

✅ Done — run the next cell for the comparison table.


In [26]:

# ── Cell 8: Results comparison table ─────────────────────────────────────────
import math

_miss2 = [v for v in ["naive_scores", "hybrid_scores"] if v not in globals()]
if _miss2:
    raise RuntimeError(f"❌ Run cell 7 first. Missing: {_miss2}")

comparison = pd.DataFrame({
    "Metric": ["Semantic Similarity ↑", "Answer Correctness ↑"],
    "Naive RAG (Vector Only)": [
        naive_scores["semantic_similarity"],
        naive_scores["answer_correctness"],
    ],
    "Nexus GraphRAG (Hybrid)": [
        hybrid_scores["semantic_similarity"],
        hybrid_scores["answer_correctness"],
    ],
})
comparison["Δ Hybrid − Naive"] = (
    comparison["Nexus GraphRAG (Hybrid)"] - comparison["Naive RAG (Vector Only)"]
).round(4)

print("\n" + "="*62)
print("🏆  FINAL COMPARISON RESULTS  🏆")
print("="*62 + "\n")
print(comparison.to_markdown(index=False))
print()

for _, row in comparison.iterrows():
    delta = row["Δ Hybrid − Naive"]
    if math.isnan(delta):
        print(f"  {row['Metric']:<32}  ⚠️  nan — check cell 7 errors")
    else:
        winner = "Nexus GraphRAG ✅" if delta >= 0 else "Naive RAG"
        arrow  = "▲" if delta >= 0 else "▼"
        print(f"  {row['Metric']:<32}  {arrow}  winner → {winner}  (Δ {delta:+.4f})")

print("\n")
print("📋 Metric explanations:")
print("  Semantic Similarity  — cosine similarity of answer vs ground-truth embeddings (0–1)")
print("  Answer Correctness   — LLM judge: factual accuracy vs ground truth (0–1)")

# Per-question answer preview
print("\n\n? Per-Question Answer Preview\n")
print(f"{'#':<3} {'Naive answer (first 80 chars)':<82} {'Hybrid answer (first 80 chars)'}")
print("─" * 170)
for i, (na, ha) in enumerate(zip(naive_answers, hybrid_answers), 1):
    n_stub = " [stub]" if len(na) < 60 else ""
    h_stub = " [stub]" if len(ha) < 60 else ""
    print(f"Q{i:<2} {(na[:79] + n_stub):<82} {ha[:79]}{h_stub}")


The history saving thread hit an unexpected error (UnicodeEncodeError('utf-8', '\n# ── Cell 8: Results comparison table ─────────────────────────────────────────\nimport math\n\n_miss2 = [v for v in ["naive_scores", "hybrid_scores"] if v not in globals()]\nif _miss2:\n    raise RuntimeError(f"❌ Run cell 7 first. Missing: {_miss2}")\n\ncomparison = pd.DataFrame({\n    "Metric": ["Semantic Similarity ↑", "Answer Correctness ↑"],\n    "Naive RAG (Vector Only)": [\n        naive_scores["semantic_similarity"],\n        naive_scores["answer_correctness"],\n    ],\n    "Nexus GraphRAG (Hybrid)": [\n        hybrid_scores["semantic_similarity"],\n        hybrid_scores["answer_correctness"],\n    ],\n})\ncomparison["Δ Hybrid − Naive"] = (\n    comparison["Nexus GraphRAG (Hybrid)"] - comparison["Naive RAG (Vector Only)"]\n).round(4)\n\nprint("\\n" + "="*62)\nprint("🏆  FINAL COMPARISON RESULTS  🏆")\nprint("="*62 + "\\n")\nprint(comparison.to_markdown(index=False))\nprint()\n\nfor _, row in compari

UnicodeEncodeError: 'utf-8' codec can't encode character '\udcdd' in position 11: surrogates not allowed